<a href="https://colab.research.google.com/github/young-suh/ESAA_2026/blob/main/week11_review.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **데이콘 x BDA 학습자 수료 예측 AI 경진대회** - private 3위


**1. 데이터 (Data)**

데이터
- 데이터 출처: BDA 8기 학습자 데이터
- 데이터 크기: 1,056개 관측치 × 46개 변수
- 독립변수: 44개
- 종속변수: withdrawal (수료 여부)
- 분석 유형: 이진 분류

주요 변수
- 학습자 특성: 학교, 전공, 직업 등
- 학습 관련 정보: 수강 강의, 학습 투입 시간 등
- 진로 관련 정보: 희망 직무, 희망 기업 등

데이터 특성
- 문자형 변수 38개, 정수형 3개, 실수형 4개, 논리형 1개
- Class 1: 730명 (69.13%)
- Class 0: 326명 (30.87%)
- 일부 변수에서 최대 99.91%의 높은 결측률 확인

전처리 방향
- 결측률 50% 초과 변수 제거
- 범주형 변수 Label Encoding
- 최빈값을 이용한 결측치 대체
- StandardScaler를 활용한 데이터 표준화

**2. 코드 흐름**

| 단계 | 분석 내용 | 사용 코드 |
|:---:|---|---|
| 1 | 데이터 불러오기 | `pd.read_csv()` |
| 2 | 변수 분리 | `drop()`, `y` |
| 3 | 결측치 확인 | `isnull().mean()` |
| 4 | 범주형 변수 인코딩 | `LabelEncoder()` |
| 5 | 결측치 대체 | `SimpleImputer()` |
| 6 | 데이터 표준화 | `StandardScaler()` |
| 7 | 변수 선택 | `SelectKBest()` |
| 8 | 데이터 분할 | `train_test_split()` |
| 9 | 모델 최적화 | `GridSearchCV()` |
| 10 | 성능 평가 및 예측 | `classification_report()` |


**3. 주요 코드 분석**

In [ ]:
# SelectKBest – 주요 변수 선택
selector = SelectKBest(score_func=f_classif, k=14)
X_selected = selector.fit_transform(X_scaled, y)

-> F-test를 이용해 각 변수가 종속변수를 구분하는 데 얼마나 유용한지 평가함

-> 점수가 높은 상위 14개 변수를 선택함

-> 불필요한 변수를 줄여 모델 학습에 활용함

In [ ]:
# SVM – 분류 모델 선정
SVC(
    class_weight={0: 1.62, 1: 0.723},
    probability=True,
    random_state=42
)

-> SVM은 두 클래스를 구분하는 최적의 결정 경계를 찾는 분류 알고리즘

-> RBF 커널을 사용해 비선형적인 분류 경계를 학습한다.

-> class_weight를 활용해 소수 클래스인 0에 더 높은 가중치를 부여한다.

In [ ]:
# GridSearchCV – 하이퍼파라미터 최적화
param_grid = {
    'C': [0.1, 1, 10],
    'gamma': ['auto', 'scale', 0.1, 0.01],
    'kernel': ['rbf']
}

grid = GridSearchCV(
    SVC(class_weight={0:1.62, 1:0.723},
        probability=True,
        random_state=42),
    param_grid=param_grid,
    scoring='balanced_accuracy',
    cv=3
)

-> C와 gamma의 여러 조합을 비교한다.

-> 3-Fold 교차검증을 통해 모델 성능을 평가한다.

-> balanced_accuracy를 기준으로 최적의 조합을 선택한다.

In [ ]:
# Threshold - 분류 기준 조정
best_threshold = 0.8

y_pred = (y_proba > best_threshold).astype(int)

-> 모델이 예측한 확률을 바탕으로 최종 클래스를 결정

-> 기본 기준인 0.5 대신 0.8을 사용

-> 기준을 높이면 Class 1로 분류하기 위한 조건이 엄격해짐

**4. 새롭게 알게 된 점 / 배운점**

새롭게 알게 된 점:
- 결측치 처리: 이전 와인 데이터에서는 결측치가 없었지만, 이번 데이터에서는 결측률에 따라 변수를 제거하고 최빈값으로 대체하는 방법을 확인함.
- 클래스 불균형 대응: class_weight와 Threshold 조정을 통해 소수 클래스의 예측 성능을 고려하는 방법을 알게 됨.
- 변수 선택: SelectKBest를 활용해 통계적 기준으로 주요 변수를 선별하는 과정을 확인함.
- 모델 최적화: GridSearchCV를 통해 SVM의 C와 gamma 등 하이퍼파라미터를 탐색하는 방법을 실제 사례로 이해함.

아쉬웠던 점:
- 데이터 누수 가능성: 학습·검증 데이터 분리 전에 표준화와 변수 선택을 진행한 점이 아쉬움.
- Threshold 설정: 0.8을 선택한 구체적인 근거와 다른 기준값과의 성능 비교가 부족함.
- 모델 성능: Accuracy 44%, Class 1 Recall 0.24로 나타나 클래스별 예측 성능의 불균형이 존재함.
- 모델 비교: SVM 외에 다른 분류 모델과의 성능 비교가 있었다면 모델 선택의 타당성을 더 명확하게 판단할 수 있었을 것임.

배울 점:
- 데이터마다 결측치와 변수의 특성이 다르므로 이에 맞는 전처리 전략을 수립해야 함.
- 클래스 불균형이 존재할 때는 Accuracy뿐 아니라 Recall, Precision, F1-score 등을 종합적으로 고려해야 함.
- 모델 학습뿐 아니라 데이터 누수 방지, 하이퍼파라미터 최적화, 평가 기준 설정까지 분석 과정 전반의 타당성을 검토해야 함.